# Logistic Regression with PCA

In this notebook we will study the [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) model on our dataset by first applying a [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) to it.

In [ ]:
#Importing libraries
import pandas as pd

#Libraries for preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
# Libraries for modeling
from sklearn.linear_model import LogisticRegression
#Libraries for evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
#Libraries for saving the model
import joblib
import os
from sklearn.pipeline import Pipeline

## Pre-processing

In [2]:
data = pd.read_csv('../data/rt_iot2022.csv')
data.head()

,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,0.281148,...,0.0,29729182.96,29729182.96,29729182.96,29729182.96,0.0,64240,26847,502,MQTT_Publish
1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,0.282277,...,0.0,29855277.06,29855277.06,29855277.06,29855277.06,0.0,64240,26847,502,MQTT_Publish
2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,0.280164,...,0.0,29842149.02,29842149.02,29842149.02,29842149.02,0.0,64240,26847,502,MQTT_Publish
3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,0.281593,...,0.0,29913774.97,29913774.97,29913774.97,29913774.97,0.0,64240,26847,502,MQTT_Publish
4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,0.282111,...,0.0,29814704.90,29814704.90,29814704.90,29814704.90,0.0,64240,26847,502,MQTT_Publish


As seen in the data analysis, we have two categorical features. We are going to encode th categories using a [OneHot Encoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [ ]:
categorical_feature = ['proto', 'service']
onehot = OneHotEncoder(sparse_output = False)
encoded_categorical = onehot.fit_transform(data[categorical_feature])

encoded_df = pd.DataFrame(encoded_categorical, columns = onehot.get_feature_names_out(categorical_feature))
data_encoded = data.drop(categorical_feature, axis = 1)
data_encoded = pd.concat([data_encoded, encoded_df], axis = 1)
data_encoded.head()

,id.orig_p,id.resp_p,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,bwd_pkts_per_sec,flow_pkts_per_sec,...,service_-,service_dhcp,service_dns,service_http,service_irc,service_mqtt,service_ntp,service_radius,service_ssh,service_ssl
0,38667,1883,32.011598,9,5,3,3,0.281148,0.156193,0.437341,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,51143,1883,31.883584,9,5,3,3,0.282277,0.156821,0.439097,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,44761,1883,32.124053,9,5,3,3,0.280164,0.155647,0.435811,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,60893,1883,31.961063,9,5,3,3,0.281593,0.156440,0.438033,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,51087,1883,31.902362,9,5,3,3,0.282111,0.156728,0.438839,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


We then separate the target from the dataset and split those two datasets in traing and testing sets.

In [ ]:
# split the data into features and target
X = data_encoded.drop('Attack_type', axis=1)
y = data_encoded['Attack_type']
# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

To ensure that our model behave correctly we standardize the data using [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## PCA and model traning

We apply the PCA in order to keep only 10 components (nearly 10% of features).  

Then we apply the Logistic Regression model with a maximum of iteration set to 1000. It will enable future optimization algorithms to run longer and get a better result.

In [ ]:
model = LogisticRegression(max_iter = 1000)
pca = PCA(n_components = 10)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)

## Evaluation of the model

As the model is really well cleaned and has a good amount of data we expect a good result on our model.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {accuracy:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

c:\Users\gaell\Documents\ml_courses\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Accuracy: 0.977, Precision: 0.977, Recall: 0.977, F1: 0.976


Indeed the result are very good ! Let's see the classification report for more detail.

In [ ]:
print(classification_report(y_test, y_pred))

c:\Users\gaell\Documents\ml_courses\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\gaell\Documents\ml_courses\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                            precision    recall  f1-score   support

            ARP_poisioning       0.92      0.76      0.83      2325
            DDOS_Slowloris       0.77      0.73      0.75       160
             DOS_SYN_Hping       1.00      1.00      1.00     28398
              MQTT_Publish       1.00      0.99      1.00      1244
Metasploit_Brute_Force_SSH       0.00      0.00      0.00        11
             NMAP_FIN_SCAN       0.00      0.00      0.00         8
         NMAP_OS_DETECTION       0.99      1.00      0.99       600
             NMAP_TCP_scan       0.98      1.00      0.99       301
             NMAP_UDP_SCAN       0.88      0.98      0.93       777
       NMAP_XMAS_TREE_SCAN       1.00      0.99      0.99       603
               Thing_Speak       0.82      0.94      0.88      2433
                Wipro_bulb       0.61      0.22      0.33        76

                  accuracy                           0.98     36936
                 macro avg       0.75      0.7

c:\Users\gaell\Documents\ml_courses\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Our model is pretty good in almost all attack categories. He is very very good at detecting : 
- Denied of Service attack using Hping tool, 
- Attaks that use enormous amounts of MQTT Publish packets,
- OS detection using nmap that enable an attacker to tailor his attack techniques for a specific OS,
- Open TCP or UDP port discovery to expose critical services.

But our model is not good at detecting :
- Attacks on Wipro lightbulb to expose wi-fi network (33% detection),
- Open port using FIN flag with Nmap (0% detection),
- Brute force attacks on SSH using Metasploit (0% detection).

## Export of model and pre-processing pipeline

To facilitate the optimization work of this model we will export both the model and the pre-processing pipeline.

In [ ]:
JOB_DIR = '../jobs'

# Define the filename for your saved model
preprocessing_filename = 'preprocessing.joblib'
PP_PATH = os.path.join(JOB_DIR, preprocessing_filename)

pipeline_steps = [
    ('scaler', scaler),
    ('pca', pca),
]

preprocessing_pipeline = Pipeline(pipeline_steps)

# --- The Export Step ---
try:
    joblib.dump(preprocessing_pipeline, os.path.abspath(PP_PATH))
    print(f"Pipeline successfully exported to: {PP_PATH}")
except Exception as e:
    print(f"An error occurred during export: {e}")

Pipeline successfully exported to: ../jobs\preprocessing.joblib


In [14]:
model_filename = 'logistic_regression_PCA_model.joblib'
MODEL_PATH = os.path.join(JOB_DIR, model_filename)
try:
    joblib.dump(model, MODEL_PATH)
    print(f"Model successfully exported to: {MODEL_PATH}")
except Exception as e:
    print(f"An error occurred during export: {e}")

Model successfully exported to: ../jobs\logistic_regression_PCA_model.joblib
